# RAG — Elasticsearch Pipeline

This notebook demonstrates a RAG pipeline using **Elasticsearch** as the search backend.

Pipeline steps:
1. **Load** FAQ documents via HTTP (`FaqHttpLoader`)
2. **Index** documents into Elasticsearch (`ElasticsearchIndex.index_docs`) — first run only
3. **Query** the pipeline with a natural-language question (`RAGBase`)
4. **Answer** is generated via OpenRouter (`OpenRouterClient`)

## Start Elasticsearch

Run Elasticsearch 8.x in Docker with a **named volume** so data survives container restarts:

```bash
docker run --rm -d \
  --name elasticsearch \
  -p 9200:9200 \
  -v es_data:/usr/share/elasticsearch/data \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  docker.elastic.co/elasticsearch/elasticsearch:8.17.6
```

Wait ~20 seconds, then verify it's up:

```bash
curl http://localhost:9200
```

> **Note:** Requires `OPENROUTER_API_KEY` in your `.env` file.

---

## Usage modes

- **First run** — run all cells including the *Ingestion* section to load and index documents.
- **Subsequent runs** — skip the *Ingestion* section; the index already exists in Elasticsearch.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, '..')

from src import FaqHttpLoader, ElasticsearchIndex, RAGBase, OpenRouterClient

## Ingestion (first run only)

Run these cells once to load documents from the FAQ API and index them into Elasticsearch.
On subsequent runs you can skip straight to the **Querying** section below — the index persists in the named volume.

In [9]:
loader = FaqHttpLoader()
docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 1208 documents


In [10]:
ingest_index = ElasticsearchIndex(host="http://localhost:9200", index_name="faq")
ingest_index.index_docs(docs)
print(f"Indexed {len(docs)} documents into Elasticsearch")

Indexed 1208 documents into Elasticsearch


## Querying

Connect to the existing index — no ingestion needed. Run from here on subsequent runs.

In [7]:
index = ElasticsearchIndex(host="http://localhost:9200", index_name="faq")
# index_docs is NOT called here — reusing the existing persistent index

assistant = RAGBase(
    index=index,
    llm=OpenRouterClient(),
    model="openrouter/owl-alpha",
    course_filter="llm-zoomcamp"
)

In [ ]:
answer = assistant.rag("How do I join the course?")
print(answer)